# Assignment 2 — Hyperparameter Tuning

**Purpose:** Tune the validation-selected architecture with repeated-seed rolling random search.

This notebook is an executable evidence view over the shared local pipeline. The
command-line runner and notebook call the same stage implementation, preventing code
drift. Completed artifacts are reused; no Google Colab or Google Drive runtime is used.


In [1]:
# Local WSL project bootstrap. Dependency installation belongs to the
# locked project environment and is intentionally not repeated in notebooks.
from pathlib import Path
import json
import os
import platform
import sys

candidate = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [candidate, *candidate.parents] if (path / "shared").exists()),
    candidate,
)
if not (PROJECT_ROOT / "shared").exists():
    raise FileNotFoundError(
        "Open this notebook from the local odl-assignment project root."
    )
os.environ["ODL_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.config import PATHS
from shared.runtime import RuntimeOptions
from shared.workflow import run_stage

OPTIONS = RuntimeOptions.from_environment()
print("Project root:", PROJECT_ROOT)
print("Run mode:", OPTIONS.mode)
print("Device policy:", OPTIONS.device)
print("WSL distribution:", os.getenv("WSL_DISTRO_NAME", "not detected"))


I0000 00:00:1786494558.344682  110400 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Project root: /home/nicholas/code/odl-assignment
Run mode: full
Device policy: gpu
WSL distribution: Ubuntu-24.04


I0000 00:00:1786494559.390955  110400 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Run the shared stage or load its verified completed result.
STAGE_SUMMARY = run_stage("tune", options=OPTIONS)
STAGE_SUMMARY


{'architecture': 'lstm',
 'candidate_count': 16,
 'validation_fits': 96,
 'selected_candidate': 'candidate_00',
 'formulation': 'direct',
 'validation_mae': 9.102579422970097,
 'epoch_budget': 11,
 'training_seconds': 156.92691240600107,
 'run_mode': 'full'}

In [3]:
# Record the exact runtime used for this output when preflight evidence exists.
environment_path = PATHS.environment_artifacts / f"environment_{OPTIONS.mode}.json"
ENVIRONMENT = (
    json.loads(environment_path.read_text(encoding="utf-8"))
    if environment_path.exists()
    else {"status": "preflight evidence not yet generated"}
)
ENVIRONMENT


{'captured_at_utc': '2026-08-11T17:27:38.163966+00:00',
 'project_root': '/home/nicholas/code/odl-assignment',
 'platform': 'Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39',
 'system': 'Linux',
 'release': '6.6.87.2-microsoft-standard-WSL2',
 'python': '3.12.3',
 'wsl_distribution': 'Ubuntu-24.04',
 'device': 'gpu',
 'tensorflow_gpu_devices': ['/physical_device:GPU:0'],
 'batch_size': 256,
 'mixed_precision': False,
 'nvidia_smi': {'available': True,
  'name': 'NVIDIA GeForce RTX 3050 Laptop GPU',
  'driver_version': '591.74',
  'memory.total': '4096',
  'memory.used': '167',
  'utilization.gpu': '3',
  'temperature.gpu': '57',
  'power.draw': '17.83'},
 'packages': {'tensorflow': '2.21.0',
  'keras-tuner': '1.4.8',
  'numpy': '2.3.5',
  'pandas': '2.3.3',
  'pyarrow': '25.0.0',
  'scikit-learn': '1.9.0',
  'matplotlib': '3.11.1',
  'seaborn': '0.13.2'},
 'git': {'available': True,
  'revision': 'c9034ba664005d472c18865f97478681c622d189',
  'dirty': True},
 'experiment_co

## Stratified random-search evidence

**Decision.** Evaluate 16 deterministic candidates across three rolling folds and seeds
42/52. Two anchors preserve the basic and previous-best settings; remaining candidates
balance direct and persistence-residual formulations. Random search is efficient when
only some dimensions strongly affect performance
([Bergstra and Bengio](https://www.jmlr.org/papers/v13/bergstra12a.html)). **Alternatives.**
The previous single-execution search result was smaller than seed variation.
**Trade-off.** Six validation fits per candidate cost more but make selection more
defensible. The frozen test benchmark is never read during selection.


In [4]:
import pandas as pd

suffix = OPTIONS.artifact_suffix
trials = pd.read_csv(
    PATHS.assignment2_tables / f"random_search_rolling_results{suffix}.csv"
)
summary = pd.read_csv(
    PATHS.assignment2_tables / f"random_search_summary{suffix}.csv"
)
best_hyperparameters = json.loads(
    (PATHS.models / f"best_hyperparameters{suffix}.json").read_text(encoding="utf-8")
)
display(summary)
display(trials)
display(best_hyperparameters)


,candidate_id,mean_mae,std_mae,evaluations,mean_best_epoch,median_best_epoch,parameters,num_layers,formulation,se_mae
0,candidate_13,9.041680,0.521704,6,7.500000,7.5,226945,2,residual,0.212985
1,candidate_10,9.054332,0.534681,6,10.666667,11.0,62273,2,residual,0.218283
2,candidate_14,9.065324,0.541307,6,13.166667,11.0,64385,2,residual,0.220988
3,candidate_12,9.068125,0.568381,6,5.333333,5.0,91201,1,residual,0.232041
4,candidate_05,9.078614,0.532582,6,15.333333,15.0,95361,1,direct,0.217426
5,candidate_09,9.078797,0.530508,6,5.166667,5.0,91201,1,residual,0.216579
6,candidate_01,9.084403,0.524895,6,7.833333,7.5,95361,1,direct,0.214288
7,candidate_15,9.087311,0.529545,6,4.666667,4.5,95361,1,residual,0.216186
8,candidate_08,9.094347,0.565665,6,6.666667,7.0,95361,1,residual,0.230932
9,candidate_00,9.102579,0.538517,6,11.333333,11.0,29249,1,direct,0.219849


,candidate_id,formulation,units,dense_units,dropout,learning_rate,l2_strength,num_layers,huber_delta,source,...,seed,n,mae,rmse,r2,smape_percent,best_epoch,epochs_recorded,parameters,training_seconds
0,candidate_00,direct,64,32,0.2,0.001000,0.00000,1,10.0,basic_anchor,...,42,51203,9.704914,17.955398,0.938140,18.859407,9,14,29249,203.929664
1,candidate_00,direct,64,32,0.2,0.001000,0.00000,1,10.0,basic_anchor,...,52,51203,9.655613,17.914268,0.938423,18.721008,11,16,29249,207.443004
2,candidate_01,direct,128,64,0.2,0.001075,0.00010,1,10.0,previous_best_anchor,...,42,51203,9.673598,17.923229,0.938362,18.825065,7,12,95361,105.905646
3,candidate_01,direct,128,64,0.2,0.001075,0.00010,1,10.0,previous_best_anchor,...,52,51203,9.662491,17.884460,0.938628,19.116666,9,14,95361,120.323058
4,candidate_02,direct,64,64,0.2,0.001195,0.00001,1,20.0,random,...,42,51203,9.757840,17.845557,0.938895,19.112420,13,18,31361,158.055346
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,candidate_13,residual,128,64,0.2,0.000560,0.00010,2,5.0,random,...,52,59962,8.498857,17.049369,0.935736,20.274867,10,15,226945,249.054603
92,candidate_14,residual,64,64,0.2,0.000414,0.00000,2,5.0,random,...,42,59962,8.518874,17.032870,0.935860,20.432761,11,16,64385,271.340006
93,candidate_14,residual,64,64,0.2,0.000414,0.00000,2,5.0,random,...,52,59962,8.488476,17.018417,0.935969,20.385354,15,20,64385,350.053889
94,candidate_15,residual,128,64,0.3,0.000933,0.00001,1,20.0,random,...,42,59962,8.558118,17.009825,0.936034,20.751794,4,9,95361,124.752469


{'architecture': 'lstm',
 'objective': 'mean_rolling_validation_mae',
 'values': {'candidate_id': 'candidate_00',
  'formulation': 'direct',
  'units': 64,
  'dense_units': 32,
  'dropout': 0.2,
  'learning_rate': 0.001,
  'l2_strength': 0.0,
  'num_layers': 1,
  'huber_delta': 10.0,
  'source': 'basic_anchor'},
 'epoch_budget': 11,
 'selection_rule': 'one-standard-error; parameters, layers, direct formulation',
 'run_mode': 'full',
 'config_hash': '1a37da563ad2c387'}

## Final validation selection and fixed official model

**Decision.** Apply the one-standard-error rule, preferring fewer parameters, fewer
layers, then a direct formulation when performance is practically tied. The selected
configuration is trained once with seed 42 on all pre-test data for the median
validation-derived epoch budget. Huber loss limits the influence of extreme spikes
([Huber](https://doi.org/10.1214/aoms/1177703732)). **Trade-off.** Simplicity may give up
a tiny mean-MAE advantage in exchange for stability and clearer attribution.


In [5]:
final_selection = json.loads(
    (PATHS.models / f"final_model_selection{suffix}.json").read_text(encoding="utf-8")
)
display(final_selection)
print("Rolling tuning and single-model selection complete.")


{'architecture': 'lstm',
 'selected_candidate': {'candidate_id': 'candidate_00',
  'formulation': 'direct',
  'units': 64,
  'dense_units': 32,
  'dropout': 0.2,
  'learning_rate': 0.001,
  'l2_strength': 0.0,
  'num_layers': 1,
  'huber_delta': 10.0,
  'source': 'basic_anchor'},
 'selected_seed': 42,
 'epoch_budget': 11,
 'selection_metric': 'mean_rolling_validation_mae',
 'mean_validation_mae': 9.102579422970097,
 'validation_standard_error': 0.21984863509991373,
 'final_checkpoint': '/home/nicholas/code/odl-assignment/artifacts/models/tuned_final.keras',
 'frozen_test_used_for_selection': False,
 'run_mode': 'full',
 'config_hash': '1a37da563ad2c387'}

Rolling tuning and single-model selection complete.
